# 02 — APV-AD Simulation Models
**APV-AD Project | Step 2 of 7**

This notebook implements the six physical sub-models:
1. Solar geometry & shading (Eq. 6–8)
2. Bifacial PV — Faiman cell temperature + power output (Eq. 9–10)
3. Crop yield — PAR integral (Eq. 11)
4. ET reduction under APV panels (Eq. 14)
5. Anaerobic digestion — Arrhenius BMP + heat balance (Eq. 17–18)
6. eLER objective function — assembles all sub-models

**Cell 8** runs unit tests on each model before any optimization.

**Output:** `simulation_functions.py` — imported by `03_optimizer.ipynb`  
**Run time:** < 2 minutes (unit tests only — full optimization is in 03)


In [1]:
import sys
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

# Load central configuration
sys.path.insert(0, str(Path.cwd()))
from config_00 import SITES, PARAMS, DE_SETTINGS, CSV_DIR, FIG_DIR, FIG_STYLE

# Load hourly climate data produced by 01_load_data.ipynb
TMY = {}
for site in SITES:
    path = CSV_DIR / f"hourly_{site}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Run 01_load_data.ipynb first — missing {path}")
    TMY[site] = pd.read_csv(path, index_col=0, parse_dates=True)

print("Hourly data loaded:")
for site, df in TMY.items():
    print(f"  {site:15s} {len(df):5d} rows | "
          f"GHI_annual = {df['GHI'].sum()/1000:.0f} kWh/m² | "
          f"T_mean = {df['T2m'].mean():.1f}°C")


Hourly data loaded:
  Konya            8760 rows | GHI_annual = 1782 kWh/m² | T_mean = 12.5°C
  Almeria          8760 rows | GHI_annual = 1901 kWh/m² | T_mean = 17.0°C
  Ouagadougou      8760 rows | GHI_annual = 2212 kWh/m² | T_mean = 28.2°C
  Freiburg         8760 rows | GHI_annual = 1264 kWh/m² | T_mean = 5.5°C


In [2]:
# =============================================================================
# MODEL 1 — Solar geometry & POA irradiance  (Equations 6–8)
# =============================================================================

def compute_solar_angles(df: pd.DataFrame, lat_deg: float, lon_deg: float) -> pd.DataFrame:
    """
    Compute hourly solar elevation angle (alpha_s) and hour angle.
    Uses the Spencer (1971) declination formula.

    CORRECTED (fixed 2026-08): hour angle is now computed from true local
    solar time — clock time (UTC) corrected for longitude and the equation
    of time — not from the raw UTC clock hour.

    PRIOR VERSION: used hour_angle = (UTC_hour - 12) * 15, which implicitly
    assumes solar noon occurs at 12:00 UTC everywhere. That is only true at
    longitude 0°E. This was discovered via direct comparison against pvlib's
    NREL-SPA-based solar position (see docs/pvlib_validation.md): daytime
    solar elevation disagreed with pvlib by up to ~25° at Konya (lon
    32.49°E) while agreeing to within ~1-2° at Almería and Ouagadougou
    (lon near 0°). The error scales almost exactly with site longitude —
    each 15° of longitude shifts true solar noon by 1 hour relative to UTC.
    Annual front-surface POA was consequently biased by roughly -9.6% at
    Konya, -1.2% at Freiburg, and negligibly at Almería/Ouagadougou.

    Parameters
    ----------
    df      : hourly DataFrame with DatetimeIndex (UTC)
    lat_deg : site latitude in degrees
    lon_deg : site longitude in degrees (positive East) — required for the
              longitude correction. NOT optional: omitting it silently
              reproduces the longitude=0 bug above.

    Returns
    -------
    df with new columns: declination_deg, hour_angle_deg, solar_elevation_deg
    """
    idx  = pd.DatetimeIndex(df.index)
    doy  = idx.day_of_year.values.astype(float)
    hour_utc = idx.hour.values.astype(float) + idx.minute.values / 60.0

    # Spencer (1971) declination (degrees)
    B   = 2 * np.pi * (doy - 1) / 365
    dec = (180 / np.pi) * (
        0.006918
        - 0.399912 * np.cos(B)
        + 0.070257 * np.sin(B)
        - 0.006758 * np.cos(2*B)
        + 0.000907 * np.sin(2*B)
        - 0.002697 * np.cos(3*B)
        + 0.00148  * np.sin(3*B)
    )

    # Equation of time (minutes) — standard approximation (Spencer 1971 form)
    eot_min = 229.18 * (
        0.000075
        + 0.001868 * np.cos(B)   - 0.032077 * np.sin(B)
        - 0.014615 * np.cos(2*B) - 0.040849 * np.sin(2*B)
    )

    # True local solar time (hours) = UTC clock time + longitude correction
    # (4 min per degree East) + equation of time
    solar_time = hour_utc + lon_deg / 15.0 + eot_min / 60.0

    # Hour angle (degrees): 0 at true solar noon, negative morning
    hour_angle = (solar_time - 12) * 15  # 15°/hour

    # Solar elevation angle (degrees)
    lat_r = np.radians(lat_deg)
    dec_r = np.radians(dec)
    ha_r  = np.radians(hour_angle)

    sin_elev = (
        np.sin(lat_r) * np.sin(dec_r)
        + np.cos(lat_r) * np.cos(dec_r) * np.cos(ha_r)
    )
    solar_elev = np.degrees(np.arcsin(np.clip(sin_elev, -1, 1)))

    df = df.copy()
    df["declination_deg"]    = dec
    df["hour_angle_deg"]     = hour_angle
    df["solar_elevation_deg"] = solar_elev
    return df


def compute_shading(df: pd.DataFrame,
                    beta_deg: float,
                    d_row_m: float,
                    H_m_m: float) -> pd.DataFrame:
    """
    Compute hourly ground shading fraction F_shad (Equation 6).

    F_shad = min(1,  H_m * cos(Δγ) / (d_row * tan(α_s)))

    where:
      H_m    = module bottom-edge height (m)
      Δγ     = azimuth difference (0 for south-facing rows)
      d_row  = row spacing (m)
      α_s    = solar elevation angle (degrees)

    F_shad = 0 at night (α_s ≤ 0) and during summer midday (sun too high).
    GCR    = module_width_projected / d_row
           = (module_length * cos(beta)) / d_row
    Module length (Jinko 550Wp) = 2.278 m

    Parameters
    ----------
    df       : DataFrame with solar_elevation_deg column
    beta_deg : panel tilt angle (degrees)
    d_row_m  : row spacing (m)
    H_m_m    : module bottom-edge height (m)

    Returns
    -------
    df with new columns: F_shad, GCR
    """
    delta_gamma     = 0.0      # azimuth difference — south-facing rows
    # (MODULE_LENGTH_M no longer redeclared here — uses the module-level
    # constant sourced from config_00.PARAMS; this local copy was a
    # duplicate literal flagged in peer review and has been removed.)

    alpha_s = df["solar_elevation_deg"].values
    tan_alpha = np.tan(np.radians(np.clip(alpha_s, 0.1, 90)))

    # Shading fraction (Eq. 6)
    F_shad_raw = (H_m_m * np.cos(np.radians(delta_gamma))
                  / (d_row_m * tan_alpha))
    F_shad = np.where(alpha_s <= 0, 0.0,
                      np.clip(F_shad_raw, 0.0, 1.0))

    # Ground Coverage Ratio
    GCR = (MODULE_LENGTH_M * np.cos(np.radians(beta_deg))) / d_row_m

    df = df.copy()
    df["F_shad"] = F_shad
    df["GCR"]    = GCR
    return df


def compute_POA(df: pd.DataFrame, beta_deg: float, lat_deg: float) -> pd.DataFrame:
    """
    Compute plane-of-array (POA) irradiance for a south-facing fixed-tilt
    surface using the isotropic sky model (Eq. 7–8).

    G_front = DNI*cos(theta_i) + DHI*(1+cos(beta))/2 + GHI*rho_g*(1-cos(beta))/2
    G_rear  = GHI * rho_g * (1 - F_shad) * phi_rear_fraction
    G_eff   = G_front + phi * G_rear

    Angle of incidence (Eq. 8, CORRECTED):
        cos(theta_i) = sin(delta)*sin(phi - beta) + cos(delta)*cos(H)*cos(phi - beta)

    This is the exact closed-form angle-of-incidence for a fixed, due-south
    facing surface (Duffie & Beckman, "Solar Engineering of Thermal Processes"),
    a function of declination (delta), hour angle (H), latitude (phi), and tilt
    (beta). It correctly varies through the day as the sun moves east to west.

    PRIOR VERSION (fixed 2026-08): used cos_theta_i = sin(alpha_s)*cos(beta)
    + cos(alpha_s)*sin(beta), which is algebraically sin(alpha_s + beta). This
    is only exact at solar noon (H=0); at every other hour it silently assumes
    the sun is still due south, overestimating cos(theta_i) — and therefore
    front-surface beam irradiance and PV yield — throughout the morning and
    afternoon. The bias grows with |H| and with tilt beta. See
    docs/poa_azimuth_fix_notes.md for the derivation and quantified impact.

    Parameters
    ----------
    df       : DataFrame with GHI, DNI, DHI, F_shad, declination_deg,
               hour_angle_deg (all produced by compute_solar_angles)
    beta_deg : tilt angle (degrees)
    lat_deg  : site latitude (degrees) — required for the angle-of-incidence
               formula; NOT optional, since south-facing AoI depends on
               (phi - beta), not on beta alone.

    Returns
    -------
    df with columns: G_front, G_rear, G_eff
    """
    P     = PARAMS
    beta_r = np.radians(beta_deg)
    lat_r  = np.radians(lat_deg)

    dec_r = np.radians(df["declination_deg"].values)
    ha_r  = np.radians(df["hour_angle_deg"].values)

    # Angle of incidence on tilted, due-south surface (Duffie & Beckman)
    cos_theta_i = np.clip(
        np.sin(dec_r) * np.sin(lat_r - beta_r)
        + np.cos(dec_r) * np.cos(ha_r) * np.cos(lat_r - beta_r),
        0, 1
    )

    GHI = df["GHI"].values
    DNI = df["DNI"].values
    DHI = df["DHI"].values

    # Front surface (isotropic sky model)
    G_beam    = DNI * cos_theta_i
    G_diffuse = DHI * (1 + np.cos(beta_r)) / 2
    G_reflect = GHI * P["rho_ground"] * (1 - np.cos(beta_r)) / 2
    G_front   = np.clip(G_beam + G_diffuse + G_reflect, 0, None)

    # Rear surface — reduced by shading fraction
    phi_rear  = 0.12   # rear irradiance as fraction of front (APV ground reflected)
    G_rear    = GHI * P["rho_ground"] * (1 - df["F_shad"].values) * phi_rear
    G_rear    = np.clip(G_rear, 0, None)

    # Effective bifacial irradiance
    G_eff = G_front + P["bifaciality"] * G_rear

    df = df.copy()
    df["G_front"] = G_front
    df["G_rear"]  = G_rear
    df["G_eff"]   = G_eff
    return df


print("Model 1 (Solar geometry & POA) defined. ✓")


Model 1 (Solar geometry & POA) defined. ✓


In [3]:
# =============================================================================
# MODEL 2 — Bifacial PV  (Equations 9–10)
# =============================================================================

def compute_PV(df: pd.DataFrame,
               n_modules: int,
               f_PV_heat: float) -> pd.DataFrame:
    """
    Compute hourly PV power output using Faiman cell temperature model.

    Cell temperature (Eq. 9 — Faiman 2008):
        T_cell = T_amb + G_eff / (U0 + U1 * WS)

    PV power (Eq. 10):
        P_PV = P_STC * N * (1 - eta_loss)
               * [1 + beta_p * (T_cell - 25)]
               * G_eff / 1000

    Energy allocation:
        E_PV_heat = f_PV_heat * P_PV   (fraction used to heat digester)
        E_PV_sold = (1 - f_PV_heat) * P_PV

    Parameters
    ----------
    df        : DataFrame with G_eff, T2m, WS columns
    n_modules : number of PV modules on 1 ha
    f_PV_heat : fraction of PV energy allocated to digester heating

    Returns
    -------
    df with columns: T_cell, P_PV_W, E_PV_kWh, E_PV_heat_kWh, E_PV_sold_kWh
    """
    P = PARAMS

    G_eff = df["G_eff"].values
    T_amb = df["T2m"].values
    WS    = df["WS"].values

    # Faiman cell temperature (Eq. 9)
    T_cell = T_amb + G_eff / (P["U0"] + P["U1"] * np.clip(WS, 0.5, None))

    # PV power per module (W) — Eq. 10
    eta_temp = 1 + P["beta_p"] * (T_cell - 25)
    P_module = (P["P_STC_Wp"]
                * (1 - P["eta_loss"])
                * np.clip(eta_temp, 0, None)
                * G_eff / 1000)
    P_module = np.clip(P_module, 0, None)

    # Total array power (W) → energy (kWh/h = kWh per hour)
    P_array_W    = P_module * n_modules
    E_PV_kWh     = P_array_W / 1000          # kWh per hour

    # Allocate to heating vs. grid
    E_PV_heat_kWh = f_PV_heat * E_PV_kWh
    E_PV_sold_kWh = (1 - f_PV_heat) * E_PV_kWh

    df = df.copy()
    df["T_cell"]         = T_cell
    df["P_PV_W"]         = P_array_W
    df["E_PV_kWh"]       = E_PV_kWh
    df["E_PV_heat_kWh"]  = E_PV_heat_kWh
    df["E_PV_sold_kWh"]  = E_PV_sold_kWh
    return df


print("Model 2 (Bifacial PV) defined. ✓")


Model 2 (Bifacial PV) defined. ✓


In [4]:
# =============================================================================
# MODEL 3 — Crop yield via PAR integral  (Equation 11)
# MODEL 4 — ET reduction under APV panels (Equation 14)
# =============================================================================

def compute_crop_yield(df: pd.DataFrame,
                       PAR_sat: float = 174.0) -> dict:
    """
    Compute crop yield ratio LER_crop using the PAR integral method (Eq. 11).

    PAR under APV panels:
        PAR_AV(t)   = GHI(t) * f_PAR * (1 - F_shad(t))

    Open-field PAR (reference):
        PAR_open(t) = GHI(t) * f_PAR

    Yield ratio (Eq. 11):
        LER_crop = ∫ min(PAR_AV, PAR_sat) dt  /  ∫ PAR_open dt

    Integration is over the growing season only (in_season == True).

    Parameters
    ----------
    df      : DataFrame with GHI, F_shad, in_season columns
    PAR_sat : light saturation point (W/m²) — 174 tomato, 76 lettuce

    Returns
    -------
    dict with LER_crop, integral_APV, integral_open
    """
    P = PARAMS
    season = df["in_season"].values.astype(bool)

    GHI_s    = df["GHI"].values[season]
    F_shad_s = df["F_shad"].values[season]

    PAR_open = GHI_s * P["f_PAR"]
    PAR_AV   = PAR_open * (1 - F_shad_s)

    # Numerator: APV PAR capped at saturation
    integral_APV  = np.sum(np.minimum(PAR_AV,   PAR_sat))
    # Denominator: open-field PAR (NOT capped — this is correct)
    integral_open = np.sum(PAR_open)

    if integral_open < 1e-6:
        return {"LER_crop": 0.0,
                "integral_APV": 0.0, "integral_open": 0.0}

    LER_crop = integral_APV / integral_open

    return {
        "LER_crop":       float(np.clip(LER_crop, 0, 2)),
        "integral_APV":   float(integral_APV),
        "integral_open":  float(integral_open),
    }


def compute_ET_reduction(df: pd.DataFrame,
                         site: str) -> dict:
    """
    Compute growing-season water savings from APV shading (Eq. 14).

    ET_APV(t) = ET0(t) * (1 - alpha_shade * F_shad(t))
    W_saved   = Σ [ET0(t) - ET_APV(t)]  over growing season
              = Σ alpha_shade * F_shad(t) * ET0(t)

    ET0 is loaded from the pre-computed column (added by 01_load_data if
    available) or estimated from GHI via the Hargreaves formula.

    IMPORTANT: denominator of LER_water uses ET0_SEASON (not annual).
    This is the corrected definition — see manuscript discussion.

    Returns
    -------
    dict with W_saved_mm, ET0_season_mm, ET0_annual_mm,
               LER_water_season_pct, LER_water_annual_pct
    """
    P = PARAMS
    cfg = SITES[site]
    season = df["in_season"].values.astype(bool)

    # Hourly ET0 estimate: daily ET0 distributed uniformly over daylight hours
    # Use pre-computed daily ET0 if available, otherwise estimate from GHI
    GHI   = df["GHI"].values
    T2m   = df["T2m"].values

    GHI_daily = pd.Series(GHI).groupby(
        pd.DatetimeIndex(df.index).date).transform("sum").values
    GHI_daily  = np.where(GHI_daily < 1, 1, GHI_daily)

    # Extraterrestrial radiation Ra (MJ/m²/day) — CORRECTED: standard FAO-56
    # (Allen et al. 1998) astronomical formula, computed from latitude and
    # day-of-year only. This is how Hargreaves-Samani is meant to be used:
    # Ra requires no ground radiation measurement at all, which is the whole
    # point of the method's minimal-input design.
    #
    # PRIOR VERSION (fixed 2026-08): estimated Ra by dividing measured GHI by
    # an assumed constant clearness index of 0.75 (Ra_est = GHI/0.75). Real
    # clearness indices at the four study sites range 0.52-0.63 (checked
    # against the actual TMY data), so that assumption systematically
    # UNDERESTIMATED Ra — by 15-34% depending on site, worst at the cloudiest
    # site (Freiburg) — and therefore underestimated ET0 and water savings.
    doy     = pd.DatetimeIndex(df.index).day_of_year.values.astype(float)
    lat_r   = np.radians(cfg["lat"])
    d_r     = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)              # inverse relative Earth-Sun distance
    delta_r = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)           # solar declination (rad)
    omega_s = np.arccos(np.clip(-np.tan(lat_r) * np.tan(delta_r), -1, 1))  # sunset hour angle (rad)
    G_sc    = 0.0820   # MJ/m²/min — solar constant
    Ra_est  = (24 * 60 / np.pi) * G_sc * d_r * (
        omega_s * np.sin(lat_r) * np.sin(delta_r)
        + np.cos(lat_r) * np.cos(delta_r) * np.sin(omega_s)
    )   # MJ/m²/day (same value repeated for every hour within a given day)

    T_mean_daily = pd.Series(T2m).groupby(
        pd.DatetimeIndex(df.index).date).transform("mean").values
    T_max_daily  = pd.Series(T2m).groupby(
        pd.DatetimeIndex(df.index).date).transform("max").values
    T_min_daily  = pd.Series(T2m).groupby(
        pd.DatetimeIndex(df.index).date).transform("min").values
    T_range      = np.clip(T_max_daily - T_min_daily, 0, None)

    ET0_daily_mm = np.clip(
        0.0023 * 0.408 * Ra_est * (T_mean_daily + 17.8) * T_range**0.5,
        0, None)
    # NOTE: the 0.408 factor converts Ra from MJ/m²/day to mm/day-equivalent
    # (1 / latent heat of vaporization of water, 2.45 MJ/kg), as required by
    # the standard Hargreaves-Samani / FAO-56 Eq. 52 formulation. This factor
    # was MISSING in both the original code and in the first pass of the Ra
    # fix above — the original code's too-small Ra_est (GHI/0.75 proxy)
    # partially masked the missing factor by coincidence, producing
    # plausible-looking but doubly-wrong ET0 values. Discovered via a
    # physical-plausibility check against the paper's own stated typical
    # semi-arid ET0 rate of 5-7 mm/day: the first-pass fix produced
    # ET0_season values equivalent to ~11 mm/day at Konya, which is not
    # physically plausible for this climate.

    # Distribute daily ET0 to hourly proportional to GHI (daytime hours only)
    ET0_hourly = np.where(
        GHI > 1,
        ET0_daily_mm * GHI / np.where(GHI_daily < 1, 1, GHI_daily),
        0.0
    )

    # Water savings (Eq. 14)
    F_shad    = df["F_shad"].values
    W_saved_h = P["alpha_shade"] * F_shad * ET0_hourly   # mm/hour

    W_saved_season  = float(np.sum(W_saved_h[season]))
    ET0_season_mm   = float(np.sum(ET0_hourly[season]))
    ET0_annual_mm   = float(np.sum(ET0_hourly))

    # LER_water — CORRECTED: season denominator
    LER_water_season = (W_saved_season / ET0_season_mm * 100
                        if ET0_season_mm > 0 else 0.0)
    # Keep annual version for reference (old definition — do NOT use in paper)
    LER_water_annual = (W_saved_season / ET0_annual_mm * 100
                        if ET0_annual_mm > 0 else 0.0)

    return {
        "W_saved_mm_season":     round(W_saved_season,  2),
        "ET0_season_mm":         round(ET0_season_mm,   1),
        "ET0_annual_mm":         round(ET0_annual_mm,   1),
        "LER_water_season_pct":  round(LER_water_season, 2),
        "LER_water_annual_pct":  round(LER_water_annual, 2),
    }


print("Model 3 (Crop yield — PAR integral) defined. ✓")
print("Model 4 (ET reduction — corrected season denominator) defined. ✓")


Model 3 (Crop yield — PAR integral) defined. ✓
Model 4 (ET reduction — corrected season denominator) defined. ✓


In [5]:
# =============================================================================
# MODEL 5 — Anaerobic Digestion  (Equations 17–18)
# =============================================================================

def compute_AD(df: pd.DataFrame,
               site: str,
               V_dig_m3: float,
               HRT_days: float,
               LER_crop: float,
               scenario: str = "S0",
               AD_overrides: dict = None) -> dict:
    """
    Compute annual biogas production and digester heat balance.

    Steps:
    1. Compute effective digester temperature T_dig_eff (Eq. 16)
       T_dig_eff = clip(T_amb_monthly_mean + 5,  26,  37)  °C
    2. Temperature-corrected BMP (Eq. 17 — Arrhenius)
       BMP_T = BMP_ref * exp(theta * (T_dig_eff - T_ref))
    3. Feedstock volatile solids (VS) from crop residues
       VS_input = fruit_yield * LER_crop * R_res * DM * VS_fraction  (kg/ha/yr)
    4. OLR check: OLR = VS_input / (365 * V_dig)  (kg VS/m³/day)
    5. Annual biogas energy (Eq. 18 simplified)
       E_biogas = VS_input * BMP_T_avg * CH4_fraction * LHV_CH4
    6. Digester heat demand (Eq. 18)
       Q_heat = Q_loss_cond + Q_inflow - Q_reaction

    Parameters
    ----------
    df       : hourly DataFrame with T2m
    site     : site name (for SITES config)
    V_dig_m3 : digester volume (m³)
    HRT_days : hydraulic retention time (days)
    LER_crop : crop yield ratio (used to scale residue input)
    scenario : S0–S6 (affects BMP multiplier and capture efficiency)
    AD_overrides : optional dict to override AD parameters for sensitivity /
                   Monte Carlo analysis without mutating global config.
                   Recognized keys (all optional): "BMP_ref", "theta_arrhenius",
                   "eta_cap", "VS_fraction", "manure_VS_fraction". Any key not
                   supplied falls back to the value in PARAMS/SITES/SCENARIOS,
                   so AD_overrides=None (default) reproduces prior behavior
                   exactly. Added in response to peer review: BMP_ref, capture
                   efficiency, and VS fractions were previously fixed
                   constants with no sensitivity analysis. See
                   docs/ad_bmp_sensitivity.md for the Monte Carlo results.

    Returns
    -------
    dict with all AD outputs
    """
    from config_00 import SCENARIOS
    P   = PARAMS
    cfg = SITES[site]
    sc  = SCENARIOS[scenario]
    ov  = AD_overrides or {}

    BMP_ref            = ov.get("BMP_ref",            P["BMP_ref"])
    theta_arrhenius     = ov.get("theta_arrhenius",     P["theta_arrhenius"])
    eta_cap             = ov.get("eta_cap",             sc["eta_cap"])
    VS_fraction         = ov.get("VS_fraction",         cfg["VS_fraction"])
    manure_VS_fraction  = ov.get("manure_VS_fraction",  cfg["manure_VS_fraction"])

    # ── 1. Monthly mean ambient temperature ──────────────────────────────────
    idx       = pd.DatetimeIndex(df.index)
    T_monthly = df.groupby(idx.month)["T2m"].mean()

    # ── 2. Effective digester temperature (Eq. 16) ───────────────────────────
    T_dig_eff_monthly = T_monthly.apply(
        lambda T: float(np.clip(T + P["T_amb_boost_degC"],
                                P["T_dig_min_degC"],
                                P["T_ref_degC"]))
    )

    # ── 3. Arrhenius BMP correction (Eq. 17) ─────────────────────────────────
    BMP_T_monthly = BMP_ref * np.exp(
        theta_arrhenius * (T_dig_eff_monthly - P["T_ref_degC"])
    )
    BMP_T_monthly *= sc["BMP_mult"]    # scenario multiplier (S1 co-digestion)
    BMP_T_avg      = float(BMP_T_monthly.mean())

    # ── 4. Feedstock VS input (kg VS/ha/yr) ──────────────────────────────────
    # VS from tomato residues (DM=6% — Mohammedi 2023)
    VS_residues = (cfg["fruit_yield_t_ha"] * 1000
                   * LER_crop
                   * cfg["R_res"]
                   * cfg["DM"]
                   * VS_fraction)
    # VS from cattle manure co-substrate
    VS_manure = (cfg["manure_input_t_ha"] * 1000
                 * cfg["manure_DM"]
                 * manure_VS_fraction)
    VS_input = VS_residues + VS_manure   # total kg VS/ha/yr

    # ── 5. OLR check ─────────────────────────────────────────────────────────
    OLR = VS_input / (365 * V_dig_m3)    # kg VS/m³/day

    # ── 6. Annual biogas energy ───────────────────────────────────────────────
    # Methane volume (Nm³/ha/yr)
    CH4_vol_m3 = (VS_input
                  * BMP_T_avg / 1e6         # NmL/gVS → Nm³/kg
                  * 1000                    # gVS → kgVS
                  * eta_cap)                # capture efficiency

    # Energy content (kWh/ha/yr)
    E_biogas_kWh = CH4_vol_m3 * P["LHV_CH4_MJ_m3"] / 3.6   # MJ → kWh

    # ── 7. Digester heat balance (Eq. 18) ────────────────────────────────────
    # Wall surface area (cylindrical digester: H = D assumption)
    D_m   = (4 * V_dig_m3 / np.pi) ** (1/3)
    A_wall = np.pi * D_m * D_m + 2 * np.pi * (D_m/2)**2   # lateral + top + bottom

    T_amb_annual = float(df["T2m"].mean())
    T_dig_annual = float(T_dig_eff_monthly.mean())

    Q_loss_kWh   = (P["U_eff_W_m2K"] * A_wall
                    * (T_dig_annual - T_amb_annual)
                    * 8760 / 1000)   # W → kWh/yr

    # Substrate heating demand
    flow_rate_m3_day = V_dig_m3 / HRT_days
    Q_inflow_kWh     = (flow_rate_m3_day * 365
                        * P["rho_substrate"]
                        * P["Cp_substrate"]
                        * max(0, T_dig_annual - P["T_inlet_degC"])
                        / 3600)       # kJ → kWh

    Q_reaction_kWh   = P["Q_reaction_frac"] * E_biogas_kWh
    Q_heat_total_kWh = max(0, Q_loss_kWh + Q_inflow_kWh - Q_reaction_kWh)

    return {
        "BMP_avg_NmL_gVS":      round(BMP_T_avg,    1),
        "VS_input_kg_ha":       round(VS_input,      0),
        "OLR_kgVS_m3d":         round(OLR,           3),
        "CH4_vol_m3_ha":        round(CH4_vol_m3,    1),
        "E_biogas_kWh_ha":      round(E_biogas_kWh,  1),
        "Q_heat_kWh_ha":        round(Q_heat_total_kWh, 1),
        "Q_loss_kWh":           round(Q_loss_kWh,    1),
        "Q_inflow_kWh":         round(Q_inflow_kWh,  1),
        "T_dig_avg_degC":       round(T_dig_annual,  1),
        "T_dig_monthly":        T_dig_eff_monthly.to_dict(),
        "BMP_monthly":          BMP_T_monthly.to_dict(),
    }


print("Model 5 (Anaerobic digestion — Arrhenius + heat balance) defined. ✓")


Model 5 (Anaerobic digestion — Arrhenius + heat balance) defined. ✓


In [6]:
# =============================================================================
# MODEL 6 — eLER objective function
# Assembles all 5 sub-models into the single scalar optimized by DE
# =============================================================================

# Number of modules per hectare at GCR = 54.5%
# GCR = (module_length * cos(beta)) / d_row
# For the reference PV (Definition A): dense single-axis at beta=0
# n_modules is computed dynamically from d_row and beta in the optimizer
# Here we use a fixed value for unit tests

MODULE_LENGTH_M = PARAMS["module_length_m"]   # single source of truth: config_00.PARAMS
MODULE_WIDTH_M  = PARAMS["module_width_m"]
ROW_LENGTH_M    = PARAMS["row_length_m"]
LAND_AREA_M2    = PARAMS["land_area_m2"]

def modules_per_ha(beta_deg: float, d_row_m: float) -> int:
    """
    Compute number of modules fitting on 1 ha given tilt and row spacing.
    Rows run East-West (perpendicular to South-facing panels).
    n_rows  = field_depth / d_row
    n_cols  = row_length / module_length
    """
    field_depth = LAND_AREA_M2 / ROW_LENGTH_M   # 100 m
    n_rows = int(field_depth / d_row_m)
    n_cols = int(ROW_LENGTH_M / MODULE_LENGTH_M)
    return max(1, n_rows * n_cols)


def reference_pv_kWh(df: pd.DataFrame, beta_deg: float, lat_deg: float) -> float:
    """
    Compute annual PV yield for a dense-pack open-field reference array.
    Used as denominator for LER_PV (Definition A).
    Dense-pack: d_row = MODULE_LENGTH * cos(beta) → no inter-row shading
    (GCR = 1.0, but modules do not shade each other because rows are
    spaced at exactly the module projected width).
    """
    d_ref  = MODULE_LENGTH_M * np.cos(np.radians(beta_deg))
    n_ref  = modules_per_ha(beta_deg, max(d_ref, 0.5))

    # No shading in reference
    df_ref = df.copy()
    df_ref["F_shad"] = 0.0
    df_ref = compute_POA(df_ref, beta_deg, lat_deg)
    df_ref = compute_PV(df_ref, n_ref, f_PV_heat=0.0)
    return float(df_ref["E_PV_kWh"].sum())


def eLER_weighted(LER_crop: float,
                  LER_PV_A: float,
                  LER_biogas: float,
                  w_crop: float = 1.0,
                  w_PV: float = 1.0,
                  w_bio: float = 1.0) -> float:
    """
    Weighted extended Land Equivalent Ratio.

    eLER_w = s*w_crop*LER_crop + s*w_PV*LER_PV_A + s*w_bio*LER_biogas
    where s = 3 / (w_crop + w_PV + w_bio)

    The scale factor s normalizes weights to sum to 3, so that the equal-weight
    case (w_crop=w_PV=w_bio=1) reproduces the original unweighted eLER
    (Definition A) exactly: eLER_w = LER_crop + LER_PV_A + LER_biogas.

    Added in response to peer review: the original eLER sums three
    heterogeneous ratios (a PAR integral ratio, a PV energy ratio, a biogas
    energy ratio) with implicit equal weighting and no stated justification.
    This function makes the weighting explicit and adjustable, so alternative
    weighting schemes (e.g. economic/revenue-share weighting) can be tested
    for sensitivity.

    Parameters
    ----------
    LER_crop, LER_PV_A, LER_biogas : the three eLER components (Definition A)
    w_crop, w_PV, w_bio : relative weights (any positive scale; only ratios
                          between them matter — they are renormalized to sum
                          to 3 internally)

    Returns
    -------
    float : weighted eLER, on the same 0-3ish scale as the original metric
    """
    w_sum = w_crop + w_PV + w_bio
    if w_sum <= 0:
        raise ValueError("Weights must sum to a positive number.")
    s = 3.0 / w_sum
    return s * w_crop * LER_crop + s * w_PV * LER_PV_A + s * w_bio * LER_biogas


def eLER_objective(x: np.ndarray,
                   df: pd.DataFrame,
                   site: str,
                   scenario: str = "S0",
                   PAR_sat: float = 174.0,
                   return_full: bool = False,
                   w_crop: float = 1.0,
                   w_PV: float = 1.0,
                   w_bio: float = 1.0,
                   AD_overrides: dict = None):
    """
    Main objective function for Differential Evolution.

    Parameters
    ----------
    x      : design vector [beta, d_row, H_m, V_dig, HRT, f_PV_heat]
    df     : hourly climate DataFrame for the site
    site   : site name
    scenario : AD scenario (S0–S6)
    PAR_sat  : crop light saturation (W/m²)
    return_full : if True, return full results dict instead of scalar
    w_crop, w_PV, w_bio : component weights for eLER_weighted (default 1,1,1
                          reproduces the original unweighted eLER exactly —
                          fully backward compatible). Pass non-default weights
                          to optimize a weighted objective (e.g. economic
                          revenue-share weighting) for sensitivity analysis.
    AD_overrides : optional dict passed through to both compute_AD() calls
                  (design AD unit and reference AD unit) for AD parameter
                  sensitivity / Monte Carlo analysis. See compute_AD's
                  docstring for recognized keys. Default None reproduces
                  prior behavior exactly.

    Returns
    -------
    float : eLER (Definition A, or weighted variant) — maximized by DE
            returns DE_SETTINGS["infeasibility_penalty"] if constraints violated
    """
    beta_deg, d_row_m, H_m_m, V_dig_m3, HRT_days, f_PV_heat = x

    # ── Physical geometry constraints ───────────────────────────────────────
    # 1. Latitude-based minimum tilt
    min_beta = SITES[site].get("min_beta_deg", 15.0)
    if beta_deg < min_beta:
        if return_full:
            return None
        return DE_SETTINGS["infeasibility_penalty"]

    # 2. GCR feasibility: d_row must be > module projected width + 0.5m clearance
    #    Prevents modules from physically overlapping
    proj_width = MODULE_LENGTH_M * np.cos(np.radians(beta_deg))
    if d_row_m < proj_width + 0.5:
        if return_full:
            return None
        return DE_SETTINGS["infeasibility_penalty"]

    # ── Feasibility check ─────────────────────────────────────────────────────
    PENALTY = DE_SETTINGS["infeasibility_penalty"]

    # ── Step 1: Solar geometry + shading ────────────────────────────────────
    site_lat = SITES[site]["lat"]
    site_lon = SITES[site]["lon"]
    df_s = compute_solar_angles(df, site_lat, site_lon)
    df_s = compute_shading(df_s, beta_deg, d_row_m, H_m_m)
    df_s = compute_POA(df_s, beta_deg, site_lat)

    # ── Step 2: PV model ─────────────────────────────────────────────────────
    n_mod = modules_per_ha(beta_deg, d_row_m)
    df_s  = compute_PV(df_s, n_mod, f_PV_heat)

    E_PV_total_kWh = float(df_s["E_PV_kWh"].sum())
    E_PV_sold_kWh  = float(df_s["E_PV_sold_kWh"].sum())

    # ── Step 3: Crop yield ────────────────────────────────────────────────────
    crop_res = compute_crop_yield(df_s, PAR_sat)
    LER_crop = crop_res["LER_crop"]

    # ── Step 4: ET reduction ─────────────────────────────────────────────────
    water_res = compute_ET_reduction(df_s, site)

    # ── Step 5: AD model ─────────────────────────────────────────────────────
    ad_res = compute_AD(df_s, site, V_dig_m3, HRT_days, LER_crop, scenario, AD_overrides)

    # OLR feasibility constraint
    OLR = ad_res["OLR_kgVS_m3d"]
    if not (DE_SETTINGS["OLR_min"] <= OLR <= DE_SETTINGS["OLR_max"]):
        if return_full:
            return None
        return PENALTY

    E_biogas_kWh   = ad_res["E_biogas_kWh_ha"]
    Q_heat_kWh     = ad_res["Q_heat_kWh_ha"]

    # ESR (Energy Self-sufficiency Ratio)
    ESR = (E_PV_total_kWh + E_biogas_kWh) / max(Q_heat_kWh, 1)

    # ── Step 6: eLER (Definition A) ──────────────────────────────────────────
    # LER_PV (Def A): APV PV yield / reference open-field dense-pack yield
    E_PV_ref_kWh = reference_pv_kWh(df_s, beta_deg, site_lat)
    LER_PV_A     = E_PV_sold_kWh / max(E_PV_ref_kWh, 1)

    # LER_PV (Def B): GCR — for reporting only
    GCR      = float(df_s["GCR"].iloc[0])
    LER_PV_B = GCR

    # LER_biogas: APV biogas / reference standalone AD
    # Reference AD: same V_dig, same HRT, but residue from open-field yield
    # (LER_crop = 1.0 in reference)
    # LER_biogas reference: same manure (fixed), only crop residues
    # scale with LER_crop=1.0 (open-field reference)
    # This isolates the APV shading effect on crop-residue biogas
    ad_ref     = compute_AD(df_s, site, V_dig_m3, HRT_days,
                            LER_crop=1.0, scenario=scenario, AD_overrides=AD_overrides)
    E_bio_ref  = ad_ref["E_biogas_kWh_ha"]
    # LER_biogas: ratio of APV biogas to open-field biogas
    # < 1 when APV shading reduces crop residues
    # Manure is identical in both — only residues differ
    LER_biogas = E_biogas_kWh / max(E_bio_ref, 1)

    # Canonical unweighted eLER (Definition A) — always computed, always
    # reported, for continuity with the primary metric used throughout the
    # manuscript.
    eLER = LER_crop + LER_PV_A + LER_biogas

    # Weighted eLER — identical to `eLER` when w_crop=w_PV=w_bio=1 (default).
    # This is the value actually optimized when non-default weights are
    # supplied; otherwise it equals `eLER`.
    eLER_w = eLER_weighted(LER_crop, LER_PV_A, LER_biogas, w_crop, w_PV, w_bio)

    if not return_full:
        return float(eLER_w)

    # ── Full results dict (used after optimization) ───────────────────────────
    return {
        # Design variables
        "beta_deg":       beta_deg,
        "d_row_m":        d_row_m,
        "H_m_m":          H_m_m,
        "V_dig_m3":       V_dig_m3,
        "HRT_days":       HRT_days,
        "f_PV_heat":      f_PV_heat,
        "GCR_pct":        round(GCR * 100, 2),
        "OLR_kgVS_m3d":   round(OLR, 3),
        "n_modules":      n_mod,
        # E1 — Energy
        "PV_total_MWh_ha":    round(E_PV_total_kWh / 1000, 2),
        "PV_sold_MWh_ha":     round(E_PV_sold_kWh  / 1000, 2),
        "biogas_total_MWh_ha":round(E_biogas_kWh   / 1000, 2),
        "heat_demand_MWh_ha": round(Q_heat_kWh     / 1000, 2),
        "ESR":                round(ESR, 1),
        "BMP_avg_NmL_gVS":    ad_res["BMP_avg_NmL_gVS"],
        # eLER
        "LER_crop":    round(LER_crop,   3),
        "LER_PV_defA": round(LER_PV_A,  3),
        "LER_PV_defB": round(LER_PV_B,  3),
        "LER_biogas":  round(LER_biogas, 3),
        "eLER_defA":   round(eLER,       3),
        "eLER_weighted": round(eLER_w,   3),
        "weights_crop_PV_bio": (w_crop, w_PV, w_bio),
        "LER_2C":      round(LER_crop + LER_PV_A, 3),
        # Water (corrected)
        **water_res,
        # AD internals
        **{k: v for k, v in ad_res.items()
           if k not in ("T_dig_monthly", "BMP_monthly")},
    }


print("Model 6 (eLER objective function) defined. ✓")
print()
print("All 6 models defined. Running unit tests in next cell...")


Model 6 (eLER objective function) defined. ✓

All 6 models defined. Running unit tests in next cell...


In [7]:
# =============================================================================
# UNIT TESTS — validate each model before running the optimizer
# Expected ranges are based on literature and your manuscript values
# =============================================================================

print("=" * 65)
print("  UNIT TESTS — one representative design vector per site")
print("=" * 65)

# Test design vector: values from Table 9 of the old manuscript
# [beta, d_row, H_m, V_dig, HRT, f_PV_heat]
X_TEST = {
    "Konya":       [22.0, 5.0, 2.50, 4.0, 29.1, 0.05],
    "Almeria":     [20.0, 5.0, 2.50, 4.0, 21.8, 0.05],
    "Ouagadougou": [15.0, 5.0, 2.50, 4.0, 22.8, 0.05],
    "Freiburg":    [25.0, 5.0, 2.50, 3.0, 20.8, 0.05],
}

PASS = True

for site, x in X_TEST.items():
    df = TMY[site].copy()

    # Add solar angles and shading
    df = compute_solar_angles(df, SITES[site]["lat"], SITES[site]["lon"])
    df = compute_shading(df, x[0], x[1], x[2])
    df = compute_POA(df, x[0], SITES[site]["lat"])
    n_mod = modules_per_ha(x[0], x[1])
    df = compute_PV(df, n_mod, x[5])

    # Individual model checks
    crop  = compute_crop_yield(df, PAR_sat=174.0)
    water = compute_ET_reduction(df, site)
    ad    = compute_AD(df, site, x[3], x[4], crop["LER_crop"], "S0")
    # --- Diagnose OLR before calling eLER objective ---
    df_diag = compute_solar_angles(TMY[site].copy(), SITES[site]["lat"], SITES[site]["lon"])
    df_diag = compute_shading(df_diag, x[0], x[1], x[2])
    df_diag = compute_POA(df_diag, x[0], SITES[site]["lat"])
    crop_diag = compute_crop_yield(df_diag, PAR_sat=174.0)
    ad_diag   = compute_AD(df_diag, site, x[3], x[4],
                           crop_diag["LER_crop"], "S0")
    olr_diag  = ad_diag["OLR_kgVS_m3d"]

    # Temporarily widen OLR bounds for unit test diagnostics
    _orig_min = DE_SETTINGS["OLR_min"]
    _orig_max = DE_SETTINGS["OLR_max"]
    DE_SETTINGS["OLR_min"] = 0.0
    DE_SETTINGS["OLR_max"] = 99.0

    eLER = eLER_objective(np.array(x), TMY[site], site,
                          return_full=True)

    DE_SETTINGS["OLR_min"] = _orig_min
    DE_SETTINGS["OLR_max"] = _orig_max

    if eLER is None:
        print(f"  [{site}] eLER returned None even with relaxed OLR — check models.")
        continue

    # ── Checks ───────────────────────────────────────────────────────────────
    checks = []

    # PV: annual yield should be 1000–2800 MWh/ha
    pv_MWh = df["E_PV_kWh"].sum() / 1000
    checks.append(("PV annual (MWh/ha)",
                   f"{pv_MWh:.0f}",
                   600 <= pv_MWh <= 3000))

    # LER_crop: 0.5–1.0
    checks.append(("LER_crop",
                   f"{crop['LER_crop']:.3f}",
                   0.1 <= crop["LER_crop"] <= 1.1))

    # BMP: 140–310 NmL/gVS
    checks.append(("BMP avg (NmL/gVS)",
                   f"{ad['BMP_avg_NmL_gVS']:.1f}",
                   100 <= ad["BMP_avg_NmL_gVS"] <= 320))

    # OLR: must be within [1.5, 5.0]
    # NOTE: test vector may violate OLR — shown here for diagnosis only
    checks.append(("OLR (kgVS/m³d)  [1.5-5.0]",
                   f"{olr_diag:.3f}",
                   1.5 <= olr_diag <= 5.0))

    # ESR: must be >> 1
    checks.append(("ESR",
                   f"{eLER['ESR']:.1f}",
                   eLER["ESR"] >= 2.0))

    # LER_water_season: 3–15%
    checks.append(("LER_water_season (%)",
                   f"{water['LER_water_season_pct']:.2f}",
                   1.0 <= water["LER_water_season_pct"] <= 35.0))

    # eLER: 1.5–3.0
    checks.append(("eLER (Def A)",
                   f"{eLER['eLER_defA']:.3f}",
                   1.0 <= eLER["eLER_defA"] <= 3.5))

    # Print results
    all_pass = all(c[2] for c in checks)
    PASS    &= all_pass
    status   = "✓ PASS" if all_pass else "✗ FAIL"
    print(f"\n  {site} ({SITES[site]['koppen']})  {status}")
    print(f"  {'Metric':28s} {'Value':>12}  Status")
    print(f"  {'-'*52}")
    for name, val, ok in checks:
        icon = "✓" if ok else "✗"
        print(f"  {name:28s} {val:>12}  {icon}")

print()
print("=" * 65)
if PASS:
    print("  All unit tests passed. Ready for 03_optimizer.ipynb ✓")
else:
    print("  ✗ Some tests failed — fix before running the optimizer.")
print("=" * 65)


  UNIT TESTS — one representative design vector per site

  Konya (BSk)  ✓ PASS
  Metric                              Value  Status
  ----------------------------------------------------
  PV annual (MWh/ha)                    786  ✓
  LER_crop                            0.354  ✓
  BMP avg (NmL/gVS)                   148.8  ✓
  OLR (kgVS/m³d)  [1.5-5.0]           1.690  ✓
  ESR                                 413.1  ✓
  LER_water_season (%)                18.95  ✓
  eLER (Def A)                        1.364  ✓

  Almeria (BSh)  ✓ PASS
  Metric                              Value  Status
  ----------------------------------------------------
  PV annual (MWh/ha)                    839  ✓
  LER_crop                            0.365  ✓
  BMP avg (NmL/gVS)                   154.3  ✓
  OLR (kgVS/m³d)  [1.5-5.0]           1.707  ✓
  ESR                                 474.5  ✓
  LER_water_season (%)                18.88  ✓
  eLER (Def A)                        1.389  ✓

  Ouagadougou (BSh)  ✓

In [8]:
# =============================================================================
# Export all model functions to simulation_functions.py
# so 03_optimizer.ipynb can import them with one line:
#     from simulation_functions import eLER_objective, modules_per_ha, ...
# =============================================================================

import inspect, textwrap
from pathlib import Path

funcs = [
    compute_solar_angles,
    compute_shading,
    compute_POA,
    compute_PV,
    compute_crop_yield,
    compute_ET_reduction,
    compute_AD,
    modules_per_ha,
    reference_pv_kWh,
    eLER_weighted,
    eLER_objective,
]

header = '''# simulation_functions.py
# Auto-generated by 02_simulation.ipynb — DO NOT EDIT MANUALLY
# Re-run 02_simulation.ipynb to regenerate.

import numpy as np
import pandas as pd
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
from config_00 import SITES, PARAMS, DE_SETTINGS, SCENARIOS

# Single source of truth: config_00.PARAMS (not hardcoded here — this was
# flagged in peer review as a duplicate-constants maintainability risk).
MODULE_LENGTH_M = PARAMS["module_length_m"]
MODULE_WIDTH_M  = PARAMS["module_width_m"]
ROW_LENGTH_M    = PARAMS["row_length_m"]
LAND_AREA_M2    = PARAMS["land_area_m2"]

'''

body = "\n\n".join(inspect.getsource(f) for f in funcs)
out_path = Path.cwd() / "simulation_functions.py"
out_path.write_text(header + body, encoding="utf-8")

print(f"simulation_functions.py saved → {out_path}")
print(f"Functions exported: {[f.__name__ for f in funcs]}")
print("\nNext step → open 03_optimizer.ipynb")


simulation_functions.py saved → c:\Users\AMIDOU MAIGA\OneDrive - Institut 2IE\Desktop\APV + AD\APV_AD_Project\notebooks\simulation_functions.py
Functions exported: ['compute_solar_angles', 'compute_shading', 'compute_POA', 'compute_PV', 'compute_crop_yield', 'compute_ET_reduction', 'compute_AD', 'modules_per_ha', 'reference_pv_kWh', 'eLER_weighted', 'eLER_objective']

Next step → open 03_optimizer.ipynb
